# Preprocessing dan Ekstraksi Fitur TSFEL - NO2 Kecamatan Bangkalan

Notebook ini melakukan tahap **preprocessing sebelum ekstraksi fitur**
sesuai instruksi Tugas 3:

1. Memuat data `NO2-Bangkalan.csv` hasil ekstraksi.
2. Deteksi outlier (metode IQR).
3. Imputasi missing value sampai seluruh data terisi.
4. Verifikasi bahwa data sudah bersih (tidak ada outlier tersisa dan
   tidak ada missing value) sebelum lanjut ke ekstraksi fitur.
5. Ekstraksi 68 fitur menggunakan **TSFEL**, dikelompokkan menjadi
   domain statistical, temporal, spectral (dan fractal, lihat catatan
   di Bagian 5).
6. Menyimpan hasil ke file CSV.


In [1]:
import inspect

import numpy as np
import pandas as pd
import tsfel.feature_extraction.features as tsfel_features


## 1. Memuat Data

Data yang dimuat adalah `NO2-Bangkalan.csv` hasil notebook
`1-ekstraksi-data-kecamatan.ipynb`, berisi kolom `date` dan `NO2`.


In [2]:
INPUT_CSV = "NO2-Bangkalan.csv"
target_pollutant = "NO2"
kecamatan_name = "Bangkalan"

df = pd.read_csv(INPUT_CSV)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# Paksa kolom target jadi numerik; nilai yang gagal dikonversi -> NaN
df[target_pollutant] = pd.to_numeric(df[target_pollutant], errors="coerce")

n_missing_before = df[target_pollutant].isna().sum()
print(f"Jumlah baris: {len(df)}")
print(f"Jumlah nilai non-numerik/kosong (sebelum deteksi outlier): {n_missing_before}")
df.head()


Jumlah baris: 132
Jumlah nilai non-numerik/kosong (sebelum deteksi outlier): 0


,date,NO2
0,2025-09-03,0.000009
1,2025-09-06,0.000051
2,2025-09-07,0.000013
3,2025-09-12,0.000024
4,2025-09-13,0.000020


## 2. Deteksi Outlier (Metode IQR)

Outlier dideteksi dengan metode **Interquartile Range (IQR)**: nilai di
luar rentang `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]` dianggap outlier dan
ditandai sebagai data hilang (NaN), untuk kemudian diisi ulang pada
tahap imputasi.


In [3]:
Q1 = df[target_pollutant].quantile(0.25)
Q3 = df[target_pollutant].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

is_outlier = (df[target_pollutant] < lower_bound) | (df[target_pollutant] > upper_bound)
n_outliers = is_outlier.sum()

print(f"Q1 = {Q1:.6e}, Q3 = {Q3:.6e}, IQR = {IQR:.6e}")
print(f"Batas bawah = {lower_bound:.6e}, batas atas = {upper_bound:.6e}")
print(f"Jumlah outlier terdeteksi: {n_outliers}")

df.loc[is_outlier, target_pollutant] = np.nan


Q1 = 2.174555e-05, Q3 = 3.719261e-05, IQR = 1.544706e-05
Batas bawah = -1.425043e-06, batas atas = 6.036320e-05
Jumlah outlier terdeteksi: 4


## 3. Imputasi Missing Value

Nilai hilang (baik dari data asli maupun hasil penandaan outlier pada
langkah sebelumnya) diisi dengan interpolasi berbasis waktu, lalu
`ffill`/`bfill` sebagai fallback untuk nilai di ujung awal/akhir deret
waktu yang tidak bisa diinterpolasi, sehingga **seluruh data terisi**
tanpa NaN tersisa.


In [4]:
df_clean = df.set_index("date").interpolate(method="time").ffill().bfill()

n_missing_after = df_clean[target_pollutant].isna().sum()
print(f"Jumlah missing value setelah imputasi: {n_missing_after}")


Jumlah missing value setelah imputasi: 0


## 4. Verifikasi Data Sudah Bersih

Sebelum lanjut ke ekstraksi fitur, dipastikan dulu tidak ada outlier
tersisa (dicek ulang dengan aturan IQR yang sama) dan tidak ada missing
value.


In [5]:
Q1_check = df_clean[target_pollutant].quantile(0.25)
Q3_check = df_clean[target_pollutant].quantile(0.75)
IQR_check = Q3_check - Q1_check
lb_check = Q1_check - 1.5 * IQR_check
ub_check = Q3_check + 1.5 * IQR_check

remaining_outliers = ((df_clean[target_pollutant] < lb_check) | (df_clean[target_pollutant] > ub_check)).sum()
remaining_missing = df_clean[target_pollutant].isna().sum()

print(f"Sisa outlier setelah imputasi : {remaining_outliers}")
print(f"Sisa missing value            : {remaining_missing}")

if remaining_outliers == 0 and remaining_missing == 0:
    print("\nData sudah bersih. Lanjut ke ekstraksi fitur TSFEL.")
else:
    print("\nPERINGATAN: data belum sepenuhnya bersih, periksa kembali sebelum lanjut.")


Sisa outlier setelah imputasi : 0
Sisa missing value            : 0

Data sudah bersih. Lanjut ke ekstraksi fitur TSFEL.


## 5. Ekstraksi 68 Fitur dengan TSFEL

Signal 1 dimensi (`NO2` yang sudah bersih) diproses dengan **68 fungsi
fitur** dari `tsfel.feature_extraction.features`, sesuai daftar yang
diberikan pada contoh kode. Setiap fitur dihitung sebagai satu nilai
skalar (rata rata jika fungsi mengembalikan banyak nilai, misalnya MFCC
atau wavelet yang mengembalikan larik koefisien).

**Frekuensi sampel (`fs`)** ditetapkan `1` (satu sampel per hari),
karena data berupa deret waktu harian.


In [6]:
fs = 1
signal_1d = df_clean[target_pollutant].astype(float).values

FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print("Jumlah fitur yang diminta:", len(FEATURE_LIST))


def to_scalar(result):
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)


def extract_one(fn_name, signal, fs):
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)


row = {}
for fn_name in FEATURE_LIST:
    row[fn_name] = extract_one(fn_name, signal_1d, fs)

extracted_features_final = pd.DataFrame([row])

print(f"Berhasil! Jumlah fitur yang dihasilkan untuk {target_pollutant}: "
      f"{extracted_features_final.shape[1]}")
extracted_features_final.head()


Jumlah fitur yang diminta: 68
Berhasil! Jumlah fitur yang dihasilkan untuk NO2: 68


C:\Users\ASUS\AppData\Local\Programs\Python\Python39\lib\site-packages\tsfel\feature_extraction\features.py:1820: UserWarning: The fractal features will not be calculated and will be replaced with 'nan' because the length of the input signal is smaller than the required minimum of 160 data points.
  warnings.warn(warning_msg, UserWarning)


,abs_energy,auc,autocorr,average_power,calc_centroid,calc_max,calc_mean,calc_median,calc_min,calc_std,...,spectral_spread,spectral_variation,spectrogram_mean_coeff,sum_abs_diff,wavelet_abs_mean,wavelet_energy,wavelet_entropy,wavelet_std,wavelet_var,zero_cross
0,1.237089e-07,0.003772,1.0,9.443428e-10,69.622301,0.000055,0.000029,0.000028,0.000004,0.000011,...,0.160823,0.610843,1.973692e-10,0.00135,0.000005,0.000016,2.137608,0.000015,2.594031e-10,0.0


### 5.1 Pengelompokan Domain Fitur

TSFEL secara resmi mengelompokkan fiturnya ke dalam **4 domain**:
`statistical`, `temporal`, `spectral`, dan `fractal` (dapat diverifikasi
dari file konfigurasi `tsfel/feature_extraction/features.json` pada
pustaka TSFEL). Karena instruksi tugas menyebut tiga domain
(statistical, temporal, spectral), fitur pada domain `fractal`
ditampilkan sebagai kelompok terpisah di bawah, dan kamu bisa
menggabungkannya ke domain `temporal` jika format laporan tugas
mengharuskan hanya tiga kelompok (fitur fraktal dihitung dari sinyal
pada domain waktu, bukan domain frekuensi).

| Domain | Jumlah Fitur | Contoh |
|--------|:---:|--------|
| Statistical | 21 | `calc_mean`, `calc_std`, `skewness`, `kurtosis`, `interq_range`, ... |
| Temporal | 15 | `autocorr`, `slope`, `zero_cross`, `mean_abs_diff`, ... |
| Spectral | 26 | `spectral_centroid`, `fundamental_frequency`, `mfcc`, `wavelet_energy`, ... |
| Fractal* | 6 | `higuchi_fractal_dimension`, `hurst_exponent`, `dfa`, ... |

\*Domain resmi ke-4 pada TSFEL; gabungkan ke `temporal` bila laporan
tugas hanya meminta tiga kelompok.


In [7]:
FEATURE_DOMAINS = {
    "statistical": [
        "abs_energy", "average_power", "calc_max", "calc_mean", "calc_median",
        "calc_min", "calc_std", "calc_var", "ecdf", "ecdf_percentile",
        "ecdf_percentile_count", "ecdf_slope", "entropy", "hist_mode",
        "interq_range", "kurtosis", "mean_abs_deviation", "median_abs_deviation",
        "pk_pk_distance", "rms", "skewness",
    ],
    "temporal": [
        "auc", "autocorr", "calc_centroid", "distance", "lempel_ziv",
        "mean_abs_diff", "mean_diff", "median_abs_diff", "median_diff",
        "negative_turning", "neighbourhood_peaks", "positive_turning",
        "slope", "sum_abs_diff", "zero_cross",
    ],
    "spectral": [
        "fundamental_frequency", "human_range_energy", "lpcc", "max_frequency",
        "max_power_spectrum", "median_frequency", "mfcc", "power_bandwidth",
        "spectral_centroid", "spectral_decrease", "spectral_distance",
        "spectral_entropy", "spectral_kurtosis", "spectral_positive_turning",
        "spectral_roll_off", "spectral_roll_on", "spectral_skewness",
        "spectral_slope", "spectral_spread", "spectral_variation",
        "spectrogram_mean_coeff", "wavelet_abs_mean", "wavelet_energy",
        "wavelet_entropy", "wavelet_std", "wavelet_var",
    ],
    "fractal": [
        "dfa", "higuchi_fractal_dimension", "hurst_exponent",
        "maximum_fractal_length", "mse", "petrosian_fractal_dimension",
    ],
}

# Verifikasi seluruh 68 fitur pada FEATURE_LIST sudah terklasifikasi
all_classified = sorted(sum(FEATURE_DOMAINS.values(), []))
missing_from_domains = sorted(set(FEATURE_LIST) - set(all_classified))
extra_in_domains = sorted(set(all_classified) - set(FEATURE_LIST))

print("Jumlah fitur per domain:")
for domain, feats in FEATURE_DOMAINS.items():
    print(f"  {domain}: {len(feats)}")
print(f"  TOTAL: {len(all_classified)}")
print()
print("Fitur pada FEATURE_LIST tapi belum terklasifikasi:", missing_from_domains)
print("Fitur pada domain tapi tidak ada di FEATURE_LIST  :", extra_in_domains)

# Buat tabel panjang (long format): satu baris per fitur, dengan kolom domain dan nilai
feature_domain_map = {feat: domain for domain, feats in FEATURE_DOMAINS.items() for feat in feats}

features_long = pd.DataFrame({
    "fitur": list(row.keys()),
    "domain": [feature_domain_map.get(f, "tidak diketahui") for f in row.keys()],
    "nilai": list(row.values()),
})
features_long = features_long.sort_values(["domain", "fitur"]).reset_index(drop=True)
features_long


Jumlah fitur per domain:
  statistical: 21
  temporal: 15
  spectral: 26
  fractal: 6
  TOTAL: 68

Fitur pada FEATURE_LIST tapi belum terklasifikasi: []
Fitur pada domain tapi tidak ada di FEATURE_LIST  : []


,fitur,domain,nilai
0,dfa,fractal,NaN
1,higuchi_fractal_dimension,fractal,NaN
2,hurst_exponent,fractal,NaN
3,maximum_fractal_length,fractal,NaN
4,mse,fractal,NaN
...,...,...,...
63,neighbourhood_peaks,temporal,5.000000e+00
64,positive_turning,temporal,4.100000e+01
65,slope,temporal,5.453864e-08
66,sum_abs_diff,temporal,1.350136e-03


## 6. Menyimpan Hasil Ekstraksi Fitur

Dua bentuk keluaran disimpan:
1. `<POLUTAN>_<Kecamatan>_TSFEL.csv`, format lebar (satu baris, 68
   kolom), sama seperti pada contoh kode.
2. `<POLUTAN>_<Kecamatan>_TSFEL_long.csv`, format panjang (satu baris
   per fitur, dengan kolom domain), memudahkan untuk dibaca ulang atau
   digabung dengan hasil kecamatan/polutan lain di kemudian hari.


In [8]:
wide_path = f"{target_pollutant}_{kecamatan_name}_TSFEL.csv"
long_path = f"{target_pollutant}_{kecamatan_name}_TSFEL_long.csv"

extracted_features_final.to_csv(wide_path, index=False)
features_long.to_csv(long_path, index=False)

print(f"Disimpan -> {wide_path} (format lebar, {extracted_features_final.shape[1]} kolom)")
print(f"Disimpan -> {long_path} (format panjang, {len(features_long)} baris)")


Disimpan -> NO2_Bangkalan_TSFEL.csv (format lebar, 68 kolom)
Disimpan -> NO2_Bangkalan_TSFEL_long.csv (format panjang, 68 baris)


## Selesai

Ringkasan yang dihasilkan notebook ini:
- Data `NO2-Bangkalan.csv` sudah bebas outlier dan missing value.
- 68 fitur TSFEL berhasil diekstraksi dan dikelompokkan ke domain
  statistical, temporal, spectral, dan fractal.
- Hasil ekstraksi fitur tersimpan pada `NO2_Bangkalan_TSFEL.csv` dan
  `NO2_Bangkalan_TSFEL_long.csv`, siap dipakai untuk tahap selanjutnya.
